 00_exploration_vllm — inspect raw vLLM benchmark outputs

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RESULTS_DIR = ROOT / "results" / "raw"

DATASETS = ["mmlu", "hellaswag", "gsm8k", "dolly"]

print("ROOT:", ROOT)
print("RESULTS_DIR:", RESULTS_DIR)
print("RESULTS_DIR exists:", RESULTS_DIR.exists())


ROOT: /home/stephane/resource-efficiency-benchmark
RESULTS_DIR: /home/stephane/resource-efficiency-benchmark/results/raw
RESULTS_DIR exists: True


 Discover vLLM runs under results/raw/<model_alias>/<run_id>/

In [2]:
run_rows = []

if RESULTS_DIR.exists():
    for model_dir in sorted(RESULTS_DIR.iterdir()):
        if not model_dir.is_dir():
            continue

        for run_dir in sorted(model_dir.iterdir()):
            if not run_dir.is_dir():
                continue

            run_rows.append({
                "model_alias_dir": model_dir.name,
                "run_id": run_dir.name,
                "run_dir": str(run_dir),
            })

runs_df = pd.DataFrame(run_rows)
runs_df.head()


,model_alias_dir,run_id,run_dir
0,gemma3_4b_w4a16,2026-03-31_15-55-49,/home/stephane/resource-efficiency-benchmark/r...
1,gemma3_4b_w4a16,2026-04-02_03-43-28,/home/stephane/resource-efficiency-benchmark/r...
2,granite4_htiny_awq,2026-03-31_15-55-49,/home/stephane/resource-efficiency-benchmark/r...
3,granite4_htiny_awq,2026-04-02_03-43-28,/home/stephane/resource-efficiency-benchmark/r...
4,llama3_2_1b,2026-03-31_15-55-49,/home/stephane/resource-efficiency-benchmark/r...


In [3]:
print("Number of vLLM runs discovered:", len(runs_df))
runs_df


Number of vLLM runs discovered: 17


,model_alias_dir,run_id,run_dir
0,gemma3_4b_w4a16,2026-03-31_15-55-49,/home/stephane/resource-efficiency-benchmark/r...
1,gemma3_4b_w4a16,2026-04-02_03-43-28,/home/stephane/resource-efficiency-benchmark/r...
2,granite4_htiny_awq,2026-03-31_15-55-49,/home/stephane/resource-efficiency-benchmark/r...
3,granite4_htiny_awq,2026-04-02_03-43-28,/home/stephane/resource-efficiency-benchmark/r...
4,llama3_2_1b,2026-03-31_15-55-49,/home/stephane/resource-efficiency-benchmark/r...
5,llama3_2_1b,2026-04-02_03-43-28,/home/stephane/resource-efficiency-benchmark/r...
6,ministral3_3b_awq,2026-03-31_15-55-49,/home/stephane/resource-efficiency-benchmark/r...
7,ministral3_3b_awq,2026-04-02_03-43-28,/home/stephane/resource-efficiency-benchmark/r...
8,ministral3_8b_awq,2026-03-31_15-55-49,/home/stephane/resource-efficiency-benchmark/r...
9,ministral3_8b_awq,2026-03-31_19-10-58,/home/stephane/resource-efficiency-benchmark/r...


 Audit expected files for each vLLM run

In [4]:
audit_rows = []

for _, r in runs_df.iterrows():
    run_dir = Path(r["run_dir"])

    record = {
        "model_alias_dir": r["model_alias_dir"],
        "run_id": r["run_id"],
        "config_yaml": (run_dir / "config.yaml").exists(),
        "server_log": (run_dir / "server.log").exists(),
    }

    for ds in DATASETS:
        record[f"{ds}_summary"] = (run_dir / f"{ds}.csv").exists()
        record[f"{ds}_samples"] = (run_dir / f"{ds}_samples.csv").exists()

    audit_rows.append(record)

audit_df = pd.DataFrame(audit_rows)
audit_df.head()


,model_alias_dir,run_id,config_yaml,server_log,mmlu_summary,mmlu_samples,hellaswag_summary,hellaswag_samples,gsm8k_summary,gsm8k_samples,dolly_summary,dolly_samples
0,gemma3_4b_w4a16,2026-03-31_15-55-49,True,True,True,True,True,True,True,True,False,False
1,gemma3_4b_w4a16,2026-04-02_03-43-28,True,True,False,False,False,False,False,False,True,True
2,granite4_htiny_awq,2026-03-31_15-55-49,True,True,True,True,True,True,True,True,False,False
3,granite4_htiny_awq,2026-04-02_03-43-28,True,True,False,False,False,False,False,False,True,True
4,llama3_2_1b,2026-03-31_15-55-49,True,True,True,True,True,True,True,True,False,False


In [5]:
audit_df


,model_alias_dir,run_id,config_yaml,server_log,mmlu_summary,mmlu_samples,hellaswag_summary,hellaswag_samples,gsm8k_summary,gsm8k_samples,dolly_summary,dolly_samples
0,gemma3_4b_w4a16,2026-03-31_15-55-49,True,True,True,True,True,True,True,True,False,False
1,gemma3_4b_w4a16,2026-04-02_03-43-28,True,True,False,False,False,False,False,False,True,True
2,granite4_htiny_awq,2026-03-31_15-55-49,True,True,True,True,True,True,True,True,False,False
3,granite4_htiny_awq,2026-04-02_03-43-28,True,True,False,False,False,False,False,False,True,True
4,llama3_2_1b,2026-03-31_15-55-49,True,True,True,True,True,True,True,True,False,False
5,llama3_2_1b,2026-04-02_03-43-28,True,True,False,False,False,False,False,False,True,True
6,ministral3_3b_awq,2026-03-31_15-55-49,True,True,True,True,True,True,True,True,False,False
7,ministral3_3b_awq,2026-04-02_03-43-28,True,True,False,False,False,False,False,False,True,True
8,ministral3_8b_awq,2026-03-31_15-55-49,True,True,True,True,True,True,True,True,False,False
9,ministral3_8b_awq,2026-03-31_19-10-58,True,True,False,False,False,False,False,False,True,True


 Compact audit summary by vLLM model alias

In [6]:
if not audit_df.empty:
    audit_summary = (
        audit_df.groupby("model_alias_dir")
        .agg(
            n_runs=("run_id", "nunique"),
            config_yaml=("config_yaml", "sum"),
            server_log=("server_log", "sum"),
            mmlu_summary=("mmlu_summary", "sum"),
            mmlu_samples=("mmlu_samples", "sum"),
            hellaswag_summary=("hellaswag_summary", "sum"),
            hellaswag_samples=("hellaswag_samples", "sum"),
            gsm8k_summary=("gsm8k_summary", "sum"),
            gsm8k_samples=("gsm8k_samples", "sum"),
            dolly_summary=("dolly_summary", "sum"),
            dolly_samples=("dolly_samples", "sum"),
        )
        .reset_index()
    )
    display(audit_summary)


,model_alias_dir,n_runs,config_yaml,server_log,mmlu_summary,mmlu_samples,hellaswag_summary,hellaswag_samples,gsm8k_summary,gsm8k_samples,dolly_summary,dolly_samples
0,gemma3_4b_w4a16,2,2,2,1,1,1,1,1,1,1,1
1,granite4_htiny_awq,2,2,2,1,1,1,1,1,1,1,1
2,llama3_2_1b,2,2,2,1,1,1,1,1,1,1,1
3,ministral3_3b_awq,2,2,2,1,1,1,1,1,1,1,1
4,ministral3_8b_awq,3,3,3,1,1,1,1,1,1,2,2
5,phi4mini_gptq_4bit,2,2,2,1,1,1,1,1,1,1,1
6,qwen2_5_1_5b,2,2,2,1,1,1,1,1,1,2,2
7,qwen2_7b_q4,1,1,1,1,1,1,1,1,1,1,1
8,qwen3_4b_awq,1,1,1,1,1,1,1,1,1,1,1


 Find potentially incomplete vLLM runs

In [7]:
if not audit_df.empty:
    missing_any_common = ~audit_df[["config_yaml", "server_log"]].all(axis=1)

    dataset_cols = []
    for ds in DATASETS:
        dataset_cols.extend([f"{ds}_summary", f"{ds}_samples"])

    missing_all_dataset_outputs = ~audit_df[dataset_cols].any(axis=1)

    incomplete_df = audit_df[missing_any_common | missing_all_dataset_outputs].copy()
    display(incomplete_df)


,model_alias_dir,run_id,config_yaml,server_log,mmlu_summary,mmlu_samples,hellaswag_summary,hellaswag_samples,gsm8k_summary,gsm8k_samples,dolly_summary,dolly_samples


 Load all vLLM summary and sample files

In [8]:
summary_tables = []
sample_tables = []

for _, r in runs_df.iterrows():
    run_dir = Path(r["run_dir"])

    for ds in DATASETS:
        summary_path = run_dir / f"{ds}.csv"
        samples_path = run_dir / f"{ds}_samples.csv"

        if summary_path.exists():
            df = pd.read_csv(summary_path)
            df["model_alias_dir"] = r["model_alias_dir"]
            df["run_id"] = r["run_id"]
            df["dataset"] = ds
            df["source_path"] = str(summary_path)
            summary_tables.append(df)

        if samples_path.exists():
            df = pd.read_csv(samples_path)
            df["model_alias_dir"] = r["model_alias_dir"]
            df["run_id"] = r["run_id"]
            df["dataset"] = ds
            df["source_path"] = str(samples_path)
            sample_tables.append(df)

summaries = pd.concat(summary_tables, ignore_index=True) if summary_tables else pd.DataFrame()
samples = pd.concat(sample_tables, ignore_index=True) if sample_tables else pd.DataFrame()

print("summaries shape:", summaries.shape)
print("samples shape:", samples.shape)


summaries shape: (38, 34)
samples shape: (31700, 34)


 Inspect raw vLLM summary files

In [9]:
summaries.head()


,Model alias,Repo id,Served model id,Accuracy mean,Accuracy sd,TTFT ms mean,TTFT ms sd,End-to-end latency ms mean,End-to-end latency ms sd,Tokens/sec mean,Tokens/sec sd,CPU percent avg mean,CPU percent avg sd,Temp C avg mean,Temp C avg sd,GPU percent avg mean,GPU percent avg sd,Power W avg mean,Power W avg sd,Sequential sustained tokens/sec mean,Sequential sustained tokens/sec sd,Sequential sustained samples/sec mean,Sequential sustained samples/sec sd,CPU percent peak,RAM used MB peak,Process RAM MB peak,Temp C peak,GPU percent peak,VRAM used MB peak,Power W peak,model_alias_dir,run_id,dataset,source_path
0,gemma3_4b_w4a16,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,0.52,0.0,71.9428,0.9192,127.9508,1.5443,18.2306,0.2049,7.4985,1.5035,70.0537,1.8942,12.8890,11.3311,60.7467,0.7789,5.5996,0.0661,5.4898,0.0648,65.0,14813.18,1019.99,79.500,29.0,6869.0,64.53,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,/home/stephane/resource-efficiency-benchmark/r...
1,gemma3_4b_w4a16,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,0.26,0.0,70.8052,1.7534,124.9218,3.0983,18.5055,0.4667,7.0284,0.6785,69.5761,0.6263,11.2297,14.2129,61.1533,0.1697,5.6406,0.1397,5.6406,0.1397,61.0,12489.36,1020.02,71.625,34.0,6802.0,63.91,gemma3_4b_w4a16,2026-03-31_15-55-49,hellaswag,/home/stephane/resource-efficiency-benchmark/r...
2,gemma3_4b_w4a16,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,0.03,0.0,71.8905,0.5932,151.7749,0.9190,44.4205,0.1135,7.4114,0.0777,72.2063,0.2803,32.8930,11.2241,101.1637,1.2629,22.2931,0.2932,4.5871,0.0603,29.3,8649.20,1020.02,74.250,100.0,6107.0,130.21,gemma3_4b_w4a16,2026-03-31_15-55-49,gsm8k,/home/stephane/resource-efficiency-benchmark/r...
3,gemma3_4b_w4a16,RedHatAI/gemma-3-4b-it-quantized.w4a16,iqbalamo93/Phi-4-mini-instruct-GPTQ-4bit,NaN,NaN,89.8702,16.4796,623.5248,48.2390,87.6739,6.5252,8.9205,2.7985,72.6397,1.0871,55.9527,35.2792,166.9069,5.0177,70.3371,6.0829,1.4846,0.1025,44.1,10372.47,1192.01,75.375,100.0,4780.0,192.33,gemma3_4b_w4a16,2026-04-02_03-43-28,dolly,/home/stephane/resource-efficiency-benchmark/r...
4,granite4_htiny_awq,cyankiwi/granite-4.0-h-tiny-AWQ-4bit,Qwen/Qwen2.5-1.5B-Instruct,0.52,0.0,68.4738,0.2988,120.6718,0.4516,19.5530,0.0631,6.4220,0.0346,69.8841,0.4322,16.7760,13.7400,62.6146,1.0045,5.9502,0.0212,5.8335,0.0208,27.3,8697.27,1021.79,71.500,30.0,6197.0,66.89,granite4_htiny_awq,2026-03-31_15-55-49,mmlu,/home/stephane/resource-efficiency-benchmark/r...


In [10]:
list(summaries.columns)


['Model alias',
 'Repo id',
 'Served model id',
 'Accuracy mean',
 'Accuracy sd',
 'TTFT ms mean',
 'TTFT ms sd',
 'End-to-end latency ms mean',
 'End-to-end latency ms sd',
 'Tokens/sec mean',
 'Tokens/sec sd',
 'CPU percent avg mean',
 'CPU percent avg sd',
 'Temp C avg mean',
 'Temp C avg sd',
 'GPU percent avg mean',
 'GPU percent avg sd',
 'Power W avg mean',
 'Power W avg sd',
 'Sequential sustained tokens/sec mean',
 'Sequential sustained tokens/sec sd',
 'Sequential sustained samples/sec mean',
 'Sequential sustained samples/sec sd',
 'CPU percent peak',
 'RAM used MB peak',
 'Process RAM MB peak',
 'Temp C peak',
 'GPU percent peak',
 'VRAM used MB peak',
 'Power W peak',
 'model_alias_dir',
 'run_id',
 'dataset',
 'source_path']

In [11]:
if not summaries.empty:
    display(
        summaries.groupby(["model_alias_dir", "dataset"])
        .size()
        .rename("n_rows")
        .reset_index()
        .sort_values(["model_alias_dir", "dataset"])
    )


,model_alias_dir,dataset,n_rows
0,gemma3_4b_w4a16,dolly,1
1,gemma3_4b_w4a16,gsm8k,1
2,gemma3_4b_w4a16,hellaswag,1
3,gemma3_4b_w4a16,mmlu,1
4,granite4_htiny_awq,dolly,1
5,granite4_htiny_awq,gsm8k,1
6,granite4_htiny_awq,hellaswag,1
7,granite4_htiny_awq,mmlu,1
8,llama3_2_1b,dolly,1
9,llama3_2_1b,gsm8k,1


 Inspect raw vLLM sample files

In [12]:
samples.head()


,repo_id,served_model_id,model_alias,dataset_name,run_idx,sample_idx,prompt,question,choices,instruction,context,gold_answer,parsed_prediction,correct,raw_output,output_tokens,ttft_ms,end_to_end_latency_ms,tokens_per_sec,cpu_percent_avg,cpu_percent_peak,ram_used_mb_peak,process_ram_mb_peak,temp_c_avg,temp_c_peak,gpu_util_percent_avg,gpu_util_percent_peak,vram_used_mb_peak,power_w_avg,power_w_peak,model_alias_dir,run_id,dataset,source_path
0,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,mmlu,1,1,"In primates, the location of which of the foll...","In primates, the location of which of the foll...",the feet || the skull || the foramen magnum ||...,NaN,NaN,2.0,C,True,C. the foramen,3,128.2590,191.7984,47.2148,11.2000,14.1,14080.68,1019.95,74.2083,74.250,0.0,0.0,6709.0,51.7133,52.39,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,/home/stephane/resource-efficiency-benchmark/r...
1,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,mmlu,1,2,In the comic strip 'Peanuts' what is Schroeder...,In the comic strip 'Peanuts' what is Schroeder...,Dancing || playing football || playing the pia...,NaN,NaN,2.0,C,True,C,1,79.7790,142.9208,15.8374,11.5667,12.6,14111.52,1019.95,74.3750,74.375,0.0,0.0,6709.0,54.2967,55.04,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,/home/stephane/resource-efficiency-benchmark/r...
2,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,mmlu,1,3,The Yang-shao culture gave way to the Lung-Sha...,The Yang-shao culture gave way to the Lung-Sha...,"6,000 B.P. || 5,000 B.P. || 4,000 B.P. || 3,00...",NaN,NaN,1.0,C,False,C,1,81.9868,147.3174,15.3068,28.0000,57.5,14117.65,1019.95,74.3750,74.375,0.0,0.0,6709.0,56.6300,57.48,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,/home/stephane/resource-efficiency-benchmark/r...
3,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,mmlu,1,4,At which of the following locations does bile ...,At which of the following locations does bile ...,Gastroesophageal sphincter || Duodenum || Ileo...,NaN,NaN,1.0,B,True,B,1,79.4288,143.5314,15.6000,27.2667,56.4,14123.26,1019.95,74.5417,74.625,0.0,0.0,6709.0,58.5467,59.84,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,/home/stephane/resource-efficiency-benchmark/r...
4,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,mmlu,1,5,"Let A be a finite set with m elements, and let...","Let A be a finite set with m elements, and let...",n^m || n!/(n - m)! || n! || n!/(m!(n - m)!),NaN,NaN,0.0,A,True,A,1,82.5121,146.4888,15.6307,11.5333,13.2,14127.28,1019.95,74.6250,74.625,0.0,0.0,6709.0,60.6300,62.17,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,/home/stephane/resource-efficiency-benchmark/r...


In [13]:
list(samples.columns)


['repo_id',
 'served_model_id',
 'model_alias',
 'dataset_name',
 'run_idx',
 'sample_idx',
 'prompt',
 'question',
 'choices',
 'instruction',
 'context',
 'gold_answer',
 'parsed_prediction',
 'correct',
 'raw_output',
 'output_tokens',
 'ttft_ms',
 'end_to_end_latency_ms',
 'tokens_per_sec',
 'cpu_percent_avg',
 'cpu_percent_peak',
 'ram_used_mb_peak',
 'process_ram_mb_peak',
 'temp_c_avg',
 'temp_c_peak',
 'gpu_util_percent_avg',
 'gpu_util_percent_peak',
 'vram_used_mb_peak',
 'power_w_avg',
 'power_w_peak',
 'model_alias_dir',
 'run_id',
 'dataset',
 'source_path']

In [14]:
if not samples.empty:
    display(
        samples.groupby(["model_alias_dir", "dataset"])
        .size()
        .rename("n_rows")
        .reset_index()
        .sort_values(["model_alias_dir", "dataset"])
    )


,model_alias_dir,dataset,n_rows
0,gemma3_4b_w4a16,dolly,100
1,gemma3_4b_w4a16,gsm8k,1000
2,gemma3_4b_w4a16,hellaswag,1000
3,gemma3_4b_w4a16,mmlu,1000
4,granite4_htiny_awq,dolly,100
5,granite4_htiny_awq,gsm8k,1000
6,granite4_htiny_awq,hellaswag,1000
7,granite4_htiny_awq,mmlu,1000
8,llama3_2_1b,dolly,100
9,llama3_2_1b,gsm8k,1000


 vLLM summary schema checks

In [15]:
if not summaries.empty:
    summary_schema = (
        summaries.groupby("dataset")
        .apply(lambda x: pd.Series({
            "n_rows": len(x),
            "n_unique_source_files": x["source_path"].nunique(),
            "columns_present": list(x.columns),
        }))
        .reset_index()
    )
    display(summary_schema)


,dataset,n_rows,n_unique_source_files,columns_present
0,dolly,11,11,"[Model alias, Repo id, Served model id, Accura..."
1,gsm8k,9,9,"[Model alias, Repo id, Served model id, Accura..."
2,hellaswag,9,9,"[Model alias, Repo id, Served model id, Accura..."
3,mmlu,9,9,"[Model alias, Repo id, Served model id, Accura..."


In [16]:
if not summaries.empty:
    summary_row_counts = (
        summaries.groupby(["model_alias_dir", "run_id", "dataset"])
        .size()
        .rename("rows_per_summary_file")
        .reset_index()
        .sort_values(["model_alias_dir", "run_id", "dataset"])
    )
    display(summary_row_counts)


,model_alias_dir,run_id,dataset,rows_per_summary_file
0,gemma3_4b_w4a16,2026-03-31_15-55-49,gsm8k,1
1,gemma3_4b_w4a16,2026-03-31_15-55-49,hellaswag,1
2,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,1
3,gemma3_4b_w4a16,2026-04-02_03-43-28,dolly,1
4,granite4_htiny_awq,2026-03-31_15-55-49,gsm8k,1
5,granite4_htiny_awq,2026-03-31_15-55-49,hellaswag,1
6,granite4_htiny_awq,2026-03-31_15-55-49,mmlu,1
7,granite4_htiny_awq,2026-04-02_03-43-28,dolly,1
8,llama3_2_1b,2026-03-31_15-55-49,gsm8k,1
9,llama3_2_1b,2026-03-31_15-55-49,hellaswag,1


In [17]:
if not summaries.empty:
    bad_summary_files = summary_row_counts[summary_row_counts["rows_per_summary_file"] != 1]
    display(bad_summary_files)


,model_alias_dir,run_id,dataset,rows_per_summary_file


 Summary null checks

In [18]:
if not summaries.empty:
    display(summaries.isna().sum().sort_values(ascending=False).head(40))


Accuracy sd                              11
Accuracy mean                            11
Repo id                                   0
Model alias                               0
Served model id                           0
TTFT ms mean                              0
TTFT ms sd                                0
End-to-end latency ms mean                0
End-to-end latency ms sd                  0
Tokens/sec mean                           0
Tokens/sec sd                             0
CPU percent avg mean                      0
CPU percent avg sd                        0
Temp C avg mean                           0
Temp C avg sd                             0
GPU percent avg mean                      0
GPU percent avg sd                        0
Power W avg mean                          0
Power W avg sd                            0
Sequential sustained tokens/sec mean      0
Sequential sustained tokens/sec sd        0
Sequential sustained samples/sec mean     0
Sequential sustained samples/sec

 vLLM-specific summary columns to verify

In [19]:
expected_vllm_summary_cols = [
    "Model alias",
    "Repo id",
    "Served model id",
    "TTFT ms mean",
    "End-to-end latency ms mean",
    "Tokens/sec mean",
    "CPU percent avg mean",
    "Process RAM MB peak",
    "GPU percent avg mean",
    "GPU percent peak",
    "VRAM used MB peak",
    "Power W avg mean",
    "Power W peak",
]

present_summary_cols = set(summaries.columns)
summary_col_check = pd.DataFrame({
    "column": expected_vllm_summary_cols,
    "present": [c in present_summary_cols for c in expected_vllm_summary_cols],
})
summary_col_check


,column,present
0,Model alias,True
1,Repo id,True
2,Served model id,True
3,TTFT ms mean,True
4,End-to-end latency ms mean,True
5,Tokens/sec mean,True
6,CPU percent avg mean,True
7,Process RAM MB peak,True
8,GPU percent avg mean,True
9,GPU percent peak,True


 Numeric sanity checks on vLLM summary metrics

In [20]:
summary_numeric_cols = [
    "Accuracy mean",
    "TTFT ms mean",
    "End-to-end latency ms mean",
    "Tokens/sec mean",
    "CPU percent avg mean",
    "CPU percent peak",
    "RAM used MB peak",
    "Process RAM MB peak",
    "Temp C avg mean",
    "Temp C peak",
    "GPU percent avg mean",
    "GPU percent peak",
    "VRAM used MB peak",
    "Power W avg mean",
    "Power W peak",
    "Sequential sustained tokens/sec mean",
    "Sequential sustained samples/sec mean",
]

summary_numeric_report = []

for col in summary_numeric_cols:
    if col in summaries.columns:
        vals = pd.to_numeric(summaries[col], errors="coerce")
        summary_numeric_report.append({
            "column": col,
            "min": vals.min(),
            "max": vals.max(),
            "n_negative": int((vals < 0).sum()),
            "n_null": int(vals.isna().sum()),
        })

pd.DataFrame(summary_numeric_report)


,column,min,max,n_negative,n_null
0,Accuracy mean,0.0300,0.5200,0,11
1,TTFT ms mean,68.3887,94.8022,0,0
2,End-to-end latency ms mean,120.6607,791.3244,0,0
3,Tokens/sec mean,17.8897,87.8898,0,0
4,CPU percent avg mean,6.3885,11.9591,0,0
5,CPU percent peak,17.4000,65.0000,0,0
6,RAM used MB peak,8610.7100,14813.1800,0,0
7,Process RAM MB peak,1016.1400,1192.4900,0,0
8,Temp C avg mean,68.4748,81.1993,0,0
9,Temp C peak,69.8750,83.7500,0,0


 vLLM sample schema checks

In [21]:
if not samples.empty:
    sample_row_counts = (
        samples.groupby(["model_alias_dir", "run_id", "dataset"])
        .size()
        .rename("rows_per_samples_file")
        .reset_index()
        .sort_values(["model_alias_dir", "run_id", "dataset"])
    )
    display(sample_row_counts)


,model_alias_dir,run_id,dataset,rows_per_samples_file
0,gemma3_4b_w4a16,2026-03-31_15-55-49,gsm8k,1000
1,gemma3_4b_w4a16,2026-03-31_15-55-49,hellaswag,1000
2,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,1000
3,gemma3_4b_w4a16,2026-04-02_03-43-28,dolly,100
4,granite4_htiny_awq,2026-03-31_15-55-49,gsm8k,1000
5,granite4_htiny_awq,2026-03-31_15-55-49,hellaswag,1000
6,granite4_htiny_awq,2026-03-31_15-55-49,mmlu,1000
7,granite4_htiny_awq,2026-04-02_03-43-28,dolly,100
8,llama3_2_1b,2026-03-31_15-55-49,gsm8k,1000
9,llama3_2_1b,2026-03-31_15-55-49,hellaswag,1000


In [22]:
expected_vllm_sample_cols = [
    "repo_id",
    "served_model_id",
    "model_alias",
    "dataset_name",
    "run_idx",
    "sample_idx",
    "output_tokens",
    "ttft_ms",
    "end_to_end_latency_ms",
    "tokens_per_sec",
    "cpu_percent_avg",
    "cpu_percent_peak",
    "process_ram_mb_peak",
    "gpu_util_percent_avg",
    "gpu_util_percent_peak",
    "vram_used_mb_peak",
    "power_w_avg",
    "power_w_peak",
]

present_sample_cols = set(samples.columns)
sample_col_check = pd.DataFrame({
    "column": expected_vllm_sample_cols,
    "present": [c in present_sample_cols for c in expected_vllm_sample_cols],
})
sample_col_check


,column,present
0,repo_id,True
1,served_model_id,True
2,model_alias,True
3,dataset_name,True
4,run_idx,True
5,sample_idx,True
6,output_tokens,True
7,ttft_ms,True
8,end_to_end_latency_ms,True
9,tokens_per_sec,True


 Per-run sample count consistency (vLLM)

In [23]:
if not samples.empty and "run_idx" in samples.columns:
    per_run_counts = (
        samples.groupby(["model_alias_dir", "run_id", "dataset", "run_idx"])
        .size()
        .rename("n_samples")
        .reset_index()
        .sort_values(["model_alias_dir", "run_id", "dataset", "run_idx"])
    )
    display(per_run_counts.head(100))


,model_alias_dir,run_id,dataset,run_idx,n_samples
0,gemma3_4b_w4a16,2026-03-31_15-55-49,gsm8k,1,100
1,gemma3_4b_w4a16,2026-03-31_15-55-49,gsm8k,2,100
2,gemma3_4b_w4a16,2026-03-31_15-55-49,gsm8k,3,100
3,gemma3_4b_w4a16,2026-03-31_15-55-49,gsm8k,4,100
4,gemma3_4b_w4a16,2026-03-31_15-55-49,gsm8k,5,100
5,gemma3_4b_w4a16,2026-03-31_15-55-49,gsm8k,6,100
6,gemma3_4b_w4a16,2026-03-31_15-55-49,gsm8k,7,100
7,gemma3_4b_w4a16,2026-03-31_15-55-49,gsm8k,8,100
8,gemma3_4b_w4a16,2026-03-31_15-55-49,gsm8k,9,100
9,gemma3_4b_w4a16,2026-03-31_15-55-49,gsm8k,10,100


In [24]:
if not samples.empty and "run_idx" in samples.columns:
    run_consistency = (
        per_run_counts.groupby(["model_alias_dir", "run_id", "dataset"])["n_samples"]
        .agg(["min", "max", "nunique"])
        .reset_index()
        .sort_values(["model_alias_dir", "run_id", "dataset"])
    )
    display(run_consistency)


,model_alias_dir,run_id,dataset,min,max,nunique
0,gemma3_4b_w4a16,2026-03-31_15-55-49,gsm8k,100,100,1
1,gemma3_4b_w4a16,2026-03-31_15-55-49,hellaswag,100,100,1
2,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu,100,100,1
3,gemma3_4b_w4a16,2026-04-02_03-43-28,dolly,10,10,1
4,granite4_htiny_awq,2026-03-31_15-55-49,gsm8k,100,100,1
5,granite4_htiny_awq,2026-03-31_15-55-49,hellaswag,100,100,1
6,granite4_htiny_awq,2026-03-31_15-55-49,mmlu,100,100,1
7,granite4_htiny_awq,2026-04-02_03-43-28,dolly,10,10,1
8,llama3_2_1b,2026-03-31_15-55-49,gsm8k,100,100,1
9,llama3_2_1b,2026-03-31_15-55-49,hellaswag,100,100,1


 Check that warmup was excluded from saved vLLM sample files

In [25]:
if not samples.empty and "run_idx" in samples.columns:
    warmup_check = (
        samples.groupby("dataset")["run_idx"]
        .agg(["min", "max", "nunique"])
        .reset_index()
    )
    display(warmup_check)


,dataset,min,max,nunique
0,dolly,1,10,10
1,gsm8k,1,10,10
2,hellaswag,1,10,10
3,mmlu,1,10,10


 Numeric sanity checks on vLLM sample metrics

In [26]:
sample_numeric_cols = [
    "output_tokens",
    "ttft_ms",
    "end_to_end_latency_ms",
    "tokens_per_sec",
    "cpu_percent_avg",
    "cpu_percent_peak",
    "ram_used_mb_peak",
    "process_ram_mb_peak",
    "temp_c_avg",
    "temp_c_peak",
    "gpu_util_percent_avg",
    "gpu_util_percent_peak",
    "vram_used_mb_peak",
    "power_w_avg",
    "power_w_peak",
]

sample_numeric_report = []

for col in sample_numeric_cols:
    if col in samples.columns:
        vals = pd.to_numeric(samples[col], errors="coerce")
        sample_numeric_report.append({
            "column": col,
            "min": vals.min(),
            "max": vals.max(),
            "n_negative": int((vals < 0).sum()),
            "n_null": int(vals.isna().sum()),
        })

pd.DataFrame(sample_numeric_report)


,column,min,max,n_negative,n_null
0,output_tokens,1.0000,93.0000,0,0
1,ttft_ms,64.9249,266.3667,0,0
2,end_to_end_latency_ms,116.0261,1150.0215,0,0
3,tokens_per_sec,13.1061,116.6058,0,0
4,cpu_percent_avg,2.8667,52.5667,0,0
5,cpu_percent_peak,4.3000,65.0000,0,0
6,ram_used_mb_peak,7472.5600,14813.1800,0,0
7,process_ram_mb_peak,1015.0700,1192.4900,0,0
8,temp_c_avg,65.0833,83.5357,0,0
9,temp_c_peak,65.1250,83.7500,0,0


 vLLM GPU / VRAM / power focused checks

In [27]:
gpu_focus_cols = [
    "gpu_util_percent_avg",
    "gpu_util_percent_peak",
    "vram_used_mb_peak",
    "power_w_avg",
    "power_w_peak",
]

gpu_focus_report = []

for col in gpu_focus_cols:
    if col in samples.columns:
        vals = pd.to_numeric(samples[col], errors="coerce")
        gpu_focus_report.append({
            "column": col,
            "min": vals.min(),
            "p50": vals.median(),
            "max": vals.max(),
            "null_rate": vals.isna().mean(),
        })

pd.DataFrame(gpu_focus_report)


,column,min,p50,max,null_rate
0,gpu_util_percent_avg,0.0000,14.00000,100.0000,0.0
1,gpu_util_percent_peak,0.0000,16.00000,100.0000,0.0
2,vram_used_mb_peak,4621.0000,6192.00000,6869.0000,0.0
3,power_w_avg,46.6967,63.65835,189.0883,0.0
4,power_w_peak,48.0800,64.49000,194.8500,0.0


 Accuracy / correctness inspection for vLLM runs

In [28]:
if "correct" in samples.columns:
    correctness_by_dataset = (
        samples.groupby("dataset")["correct"]
        .agg(["count", "mean"])
        .reset_index()
    )
    display(correctness_by_dataset)


,dataset,count,mean
0,dolly,0,NaN
1,gsm8k,9000,0.03
2,hellaswag,9000,0.26
3,mmlu,9000,0.52


In [29]:
if "parsed_prediction" in samples.columns:
    parsed_prediction_coverage = (
        samples.groupby("dataset")["parsed_prediction"]
        .apply(lambda s: s.notna().mean())
        .rename("parsed_prediction_non_null_rate")
        .reset_index()
    )
    display(parsed_prediction_coverage)


,dataset,parsed_prediction_non_null_rate
0,dolly,0.0
1,gsm8k,0.8
2,hellaswag,1.0
3,mmlu,1.0


 vLLM identity fields inspection

In [30]:
identity_cols = [c for c in ["Model alias", "Repo id", "Served model id"] if c in summaries.columns]
if identity_cols:
    display(summaries[identity_cols + ["model_alias_dir", "run_id", "dataset"]].head(20))


,Model alias,Repo id,Served model id,model_alias_dir,run_id,dataset
0,gemma3_4b_w4a16,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu
1,gemma3_4b_w4a16,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,2026-03-31_15-55-49,hellaswag
2,gemma3_4b_w4a16,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,2026-03-31_15-55-49,gsm8k
3,gemma3_4b_w4a16,RedHatAI/gemma-3-4b-it-quantized.w4a16,iqbalamo93/Phi-4-mini-instruct-GPTQ-4bit,gemma3_4b_w4a16,2026-04-02_03-43-28,dolly
4,granite4_htiny_awq,cyankiwi/granite-4.0-h-tiny-AWQ-4bit,Qwen/Qwen2.5-1.5B-Instruct,granite4_htiny_awq,2026-03-31_15-55-49,mmlu
5,granite4_htiny_awq,cyankiwi/granite-4.0-h-tiny-AWQ-4bit,Qwen/Qwen2.5-1.5B-Instruct,granite4_htiny_awq,2026-03-31_15-55-49,hellaswag
6,granite4_htiny_awq,cyankiwi/granite-4.0-h-tiny-AWQ-4bit,Qwen/Qwen2.5-1.5B-Instruct,granite4_htiny_awq,2026-03-31_15-55-49,gsm8k
7,granite4_htiny_awq,cyankiwi/granite-4.0-h-tiny-AWQ-4bit,iqbalamo93/Phi-4-mini-instruct-GPTQ-4bit,granite4_htiny_awq,2026-04-02_03-43-28,dolly
8,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,Qwen/Qwen2.5-1.5B-Instruct,llama3_2_1b,2026-03-31_15-55-49,mmlu
9,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,Qwen/Qwen2.5-1.5B-Instruct,llama3_2_1b,2026-03-31_15-55-49,hellaswag


In [31]:
identity_cols_samples = [c for c in ["model_alias", "repo_id", "served_model_id"] if c in samples.columns]
if identity_cols_samples:
    display(samples[identity_cols_samples + ["model_alias_dir", "run_id", "dataset"]].head(20))


,model_alias,repo_id,served_model_id,model_alias_dir,run_id,dataset
0,gemma3_4b_w4a16,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu
1,gemma3_4b_w4a16,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu
2,gemma3_4b_w4a16,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu
3,gemma3_4b_w4a16,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu
4,gemma3_4b_w4a16,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu
5,gemma3_4b_w4a16,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu
6,gemma3_4b_w4a16,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu
7,gemma3_4b_w4a16,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu
8,gemma3_4b_w4a16,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu
9,gemma3_4b_w4a16,RedHatAI/gemma-3-4b-it-quantized.w4a16,Qwen/Qwen2.5-1.5B-Instruct,gemma3_4b_w4a16,2026-03-31_15-55-49,mmlu


 Dolly special case in vLLM outputs

In [32]:
dolly_summary = summaries[summaries["dataset"] == "dolly"].copy()
print("vLLM dolly summary shape:", dolly_summary.shape)
display(dolly_summary.head())


vLLM dolly summary shape: (11, 34)


,Model alias,Repo id,Served model id,Accuracy mean,Accuracy sd,TTFT ms mean,TTFT ms sd,End-to-end latency ms mean,End-to-end latency ms sd,Tokens/sec mean,Tokens/sec sd,CPU percent avg mean,CPU percent avg sd,Temp C avg mean,Temp C avg sd,GPU percent avg mean,GPU percent avg sd,Power W avg mean,Power W avg sd,Sequential sustained tokens/sec mean,Sequential sustained tokens/sec sd,Sequential sustained samples/sec mean,Sequential sustained samples/sec sd,CPU percent peak,RAM used MB peak,Process RAM MB peak,Temp C peak,GPU percent peak,VRAM used MB peak,Power W peak,model_alias_dir,run_id,dataset,source_path
3,gemma3_4b_w4a16,RedHatAI/gemma-3-4b-it-quantized.w4a16,iqbalamo93/Phi-4-mini-instruct-GPTQ-4bit,NaN,NaN,89.8702,16.4796,623.5248,48.2390,87.6739,6.5252,8.9205,2.7985,72.6397,1.0871,55.9527,35.2792,166.9069,5.0177,70.3371,6.0829,1.4846,0.1025,44.1,10372.47,1192.01,75.375,100.0,4780.0,192.33,gemma3_4b_w4a16,2026-04-02_03-43-28,dolly,/home/stephane/resource-efficiency-benchmark/r...
7,granite4_htiny_awq,cyankiwi/granite-4.0-h-tiny-AWQ-4bit,iqbalamo93/Phi-4-mini-instruct-GPTQ-4bit,NaN,NaN,89.2902,16.7536,622.7314,48.8382,87.7499,6.6700,8.1615,1.8837,75.8959,1.2832,60.5007,19.0170,170.0646,4.5590,70.4746,6.1004,1.4877,0.1051,34.3,9041.88,1192.37,80.125,100.0,4804.0,194.46,granite4_htiny_awq,2026-04-02_03-43-28,dolly,/home/stephane/resource-efficiency-benchmark/r...
11,llama3_2_1b,unsloth/Llama-3.2-1B-Instruct,iqbalamo93/Phi-4-mini-instruct-GPTQ-4bit,NaN,NaN,89.7081,16.7119,622.0014,49.2200,87.8898,6.4866,8.2513,2.1234,73.9144,1.0134,40.3768,27.9111,169.0618,4.5594,70.5103,5.9820,1.4886,0.1053,53.3,9153.93,1192.16,77.875,100.0,4812.0,193.73,llama3_2_1b,2026-04-02_03-43-28,dolly,/home/stephane/resource-efficiency-benchmark/r...
15,ministral3_3b_awq,cyankiwi/Ministral-3-3B-Instruct-2512-AWQ-4bit,iqbalamo93/Phi-4-mini-instruct-GPTQ-4bit,NaN,NaN,89.4726,16.6074,622.9091,48.9443,87.7843,6.4556,8.3522,2.0912,75.1345,1.5246,55.2302,29.3406,169.7120,4.5950,70.4523,6.0321,1.4873,0.1045,44.8,9531.02,1192.23,79.750,100.0,4812.0,193.80,ministral3_3b_awq,2026-04-02_03-43-28,dolly,/home/stephane/resource-efficiency-benchmark/r...
19,ministral3_8b_awq,cyankiwi/Ministral-3-8B-Instruct-2512-AWQ-4bit,Qwen/Qwen2.5-1.5B-Instruct,NaN,NaN,79.1367,1.5744,784.8442,44.5218,78.4625,1.6341,11.9591,1.4116,81.1993,0.4148,80.4818,7.2191,171.6934,1.0160,68.1054,1.5093,1.1892,0.0642,40.3,10706.97,1022.02,83.750,100.0,6482.0,184.18,ministral3_8b_awq,2026-03-31_19-10-58,dolly,/home/stephane/resource-efficiency-benchmark/r...


In [33]:
dolly_samples = samples[samples["dataset"] == "dolly"].copy()
print("vLLM dolly samples shape:", dolly_samples.shape)
display(dolly_samples.head())


vLLM dolly samples shape: (4700, 34)


,repo_id,served_model_id,model_alias,dataset_name,run_idx,sample_idx,prompt,question,choices,instruction,context,gold_answer,parsed_prediction,correct,raw_output,output_tokens,ttft_ms,end_to_end_latency_ms,tokens_per_sec,cpu_percent_avg,cpu_percent_peak,ram_used_mb_peak,process_ram_mb_peak,temp_c_avg,temp_c_peak,gpu_util_percent_avg,gpu_util_percent_peak,vram_used_mb_peak,power_w_avg,power_w_peak,model_alias_dir,run_id,dataset,source_path
3000,RedHatAI/gemma-3-4b-it-quantized.w4a16,iqbalamo93/Phi-4-mini-instruct-GPTQ-4bit,gemma3_4b_w4a16,dolly,1,1,Identify which instrument is string or percuss...,NaN,NaN,Identify which instrument is string or percuss...,NaN,NaN,NaN,NaN,"Tifa is a percussion instrument, while Pipa is...",11,74.9472,199.7196,88.1605,14.1667,17.1,9127.23,1190.3,72.8750,72.875,70.6667,100.0,4652.0,65.3967,72.23,gemma3_4b_w4a16,2026-04-02_03-43-28,dolly,/home/stephane/resource-efficiency-benchmark/r...
3001,RedHatAI/gemma-3-4b-it-quantized.w4a16,iqbalamo93/Phi-4-mini-instruct-GPTQ-4bit,gemma3_4b_w4a16,dolly,1,2,Give me a list of the Mercury Prize winning al...,NaN,NaN,Give me a list of the Mercury Prize winning al...,NaN,NaN,NaN,NaN,"Sure, here is a list of the Mercury Prize winn...",42,76.1454,749.3013,62.3927,16.2000,18.4,9202.47,1190.3,73.0208,73.125,12.0000,12.0,4652.0,118.8867,156.51,gemma3_4b_w4a16,2026-04-02_03-43-28,dolly,/home/stephane/resource-efficiency-benchmark/r...
3002,RedHatAI/gemma-3-4b-it-quantized.w4a16,iqbalamo93/Phi-4-mini-instruct-GPTQ-4bit,gemma3_4b_w4a16,dolly,1,3,What should I do on a free afternoon in San Fr...,NaN,NaN,What should I do on a free afternoon in San Fr...,NaN,NaN,NaN,NaN,"On a free afternoon in San Francisco, you have...",63,75.1519,751.1409,93.1968,14.1000,17.3,9236.10,1190.3,73.3542,73.750,12.0000,12.0,4651.0,169.0150,175.70,gemma3_4b_w4a16,2026-04-02_03-43-28,dolly,/home/stephane/resource-efficiency-benchmark/r...
3003,RedHatAI/gemma-3-4b-it-quantized.w4a16,iqbalamo93/Phi-4-mini-instruct-GPTQ-4bit,gemma3_4b_w4a16,dolly,1,4,Give me a list of actresses who won an Oscar i...,NaN,NaN,Give me a list of actresses who won an Oscar i...,NaN,NaN,NaN,NaN,"Sure, here is a list of actresses who won an O...",43,84.1251,685.0390,71.5577,17.5600,28.9,9267.85,1190.3,73.8000,73.875,12.0000,12.0,4649.0,167.9860,176.33,gemma3_4b_w4a16,2026-04-02_03-43-28,dolly,/home/stephane/resource-efficiency-benchmark/r...
3004,RedHatAI/gemma-3-4b-it-quantized.w4a16,iqbalamo93/Phi-4-mini-instruct-GPTQ-4bit,gemma3_4b_w4a16,dolly,1,5,Give me a list of the ways you can be 'out' in...,NaN,NaN,Give me a list of the ways you can be 'out' in...,NaN,NaN,NaN,NaN,"In cricket, being 'out' refers to a player bei...",55,70.9608,672.0879,91.4948,9.8400,14.7,9300.41,1190.3,73.7500,73.875,12.0000,12.0,4645.0,176.4700,188.17,gemma3_4b_w4a16,2026-04-02_03-43-28,dolly,/home/stephane/resource-efficiency-benchmark/r...


 Final quick vLLM exploration summary

In [34]:
if not summaries.empty:
    print("vLLM model aliases discovered:", sorted(summaries["model_alias_dir"].dropna().unique()))
    print("vLLM datasets discovered:", sorted(summaries["dataset"].dropna().unique()))
    print("vLLM run_ids discovered:", summaries["run_id"].nunique())

if not samples.empty:
    sample_summary = (
        samples.groupby(["model_alias_dir", "dataset"])
        .size()
        .rename("n_sample_rows")
        .reset_index()
        .sort_values(["model_alias_dir", "dataset"])
    )
    display(sample_summary)

vLLM model aliases discovered: ['gemma3_4b_w4a16', 'granite4_htiny_awq', 'llama3_2_1b', 'ministral3_3b_awq', 'ministral3_8b_awq', 'phi4mini_gptq_4bit', 'qwen2_5_1_5b', 'qwen2_7b_q4', 'qwen3_4b_awq']
vLLM datasets discovered: ['dolly', 'gsm8k', 'hellaswag', 'mmlu']
vLLM run_ids discovered: 4


,model_alias_dir,dataset,n_sample_rows
0,gemma3_4b_w4a16,dolly,100
1,gemma3_4b_w4a16,gsm8k,1000
2,gemma3_4b_w4a16,hellaswag,1000
3,gemma3_4b_w4a16,mmlu,1000
4,granite4_htiny_awq,dolly,100
5,granite4_htiny_awq,gsm8k,1000
6,granite4_htiny_awq,hellaswag,1000
7,granite4_htiny_awq,mmlu,1000
8,llama3_2_1b,dolly,100
9,llama3_2_1b,gsm8k,1000
